In [1]:
from transformers import AutoTokenizer,AutoModelForCausalLM,TrainingArguments
from datasets import load_dataset
from peft import LoraConfig,get_peft_model

In [2]:
## 什么是模版 https://zhuanlan.zhihu.com/p/24450011474 ；https://zhuanlan.zhihu.com/p/17052593700
def format_prompt(example):
  chat = [
        {"role": "system", "content": "你是一个非常棒的人工智能助手，是崔学正开发的"},
        {"role": "user", "content": example["input"]},
        {"role": "assistant", "content": example["target"]}
    ]
  prompt=tokenizer.apply_chat_template(chat,tokenize=False)
  return {"text":prompt}

tokenizer=AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
dataset = load_dataset("YeungNLP/firefly-train-1.1M", split="train[:500]")
dataset=dataset.map(format_prompt,remove_columns=dataset.column_names)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


firefly-train-1.1M.jsonl:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1649399 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [35]:
dataset

Dataset({
    features: ['text'],
    num_rows: 500
})

In [36]:
dataset[1]

{'text': '<|im_start|>system\n你是一个非常棒的人工智能助手，是崔学正开发的<|im_end|>\n<|im_start|>user\n在上海的苹果代工厂，较低的基本工资让工人们形成了“软强制”的加班默契。加班能多拿两三千，“自愿”加班成为常态。律师提示，加班后虽能获得一时不错的报酬，但过重的工作负荷会透支身体，可能对今后劳动权利造成不利影响。\n输出摘要：<|im_end|>\n<|im_start|>assistant\n苹果代工厂员工调查：为何争着“自愿”加班<|im_end|>\n'}

In [3]:
model=AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct",device_map="auto") # device_map参数 https://zhuanlan.zhihu.com/p/606061177
tokenizer=AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
tokenizer.padding_side="left"

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [4]:
peft_config=LoraConfig(
    lora_alpha=32,
    lora_dropout=0.1, # 可以理解为dropout层缓解过拟合
    r=64,
    bias="none",# 是否训练模型的偏执项
    task_type="CAUSAL_LM",# 不同的任务类型会影响哪些部分应用lora
    target_modules=['k_proj','v_proj','q_proj']
)
model=get_peft_model(model,peft_config)

In [5]:
output_dir="./result"
training_arguments=TrainingArguments(# https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/trainer#transformers.TrainingArguments
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,# batchsize=2*4，反向传播之前积累多少步
    optim="adamw_torch",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    num_train_epochs=1,# 训练的轮数
    logging_steps=10,
    fp16=True,# 是否使用单精度
    gradient_checkpointing=True #https://zhuanlan.zhihu.com/p/1954366455685583235 时间换空间
)

In [ ]:
!pip install git+https://github.com/huggingface/trl.git

In [8]:
from trl import SFTTrainer,SFTConfig

# Set supervised fine-tuning parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=training_arguments,
    peft_config=peft_config,
)
# Train model
trainer.train()
# Save QLoRA weights
trainer.model.save_pretrained("qwen2.5-0.5b-instruct-cxz") # lora的权重就是AB的低秩矩阵

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: cxz22syx (cxz22syx-shandong-normal-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
10,3.725000
20,3.402000
30,2.994600
40,2.785300
50,2.893600
60,2.854200


In [26]:
print(trainer.model.print_trainable_parameters())

trainable params: 5,898,240 || all params: 499,931,008 || trainable%: 1.1798
None


In [10]:
from peft import AutoPeftModelForCausalLM

model = AutoPeftModelForCausalLM.from_pretrained( #
    "qwen2.5-0.5b-instruct-cxz",
    low_cpu_mem_usage=True,
    device_map="auto",
)

# Merge LoRA and base model
merged_model = model.merge_and_unload() # https://zhuanlan.zhihu.com/p/683583816 为啥要merge

In [12]:
out_dir='./result1'
merged_model.save_pretrained(out_dir, safe_serialization=True)
tokenizer.save_pretrained(out_dir)

('./result1/tokenizer_config.json',
 './result1/special_tokens_map.json',
 './result1/chat_template.jinja',
 './result1/vocab.json',
 './result1/merges.txt',
 './result1/added_tokens.json',
 './result1/tokenizer.json')

In [32]:
from transformers import pipeline

pipe = pipeline(task="text-generation", model=merged_model, tokenizer=tokenizer)

prompt_example = """<|im_start|>system
你是一个非常棒的人工智能助手，是UP主 “用代码打点酱油的chaofa” 开发的。<|im_end|>
<|im_start|>user
天气太热了，所以我今天没有学习一点。
翻译成文言文：<|im_end|>
<|im_start|>assistant
"""

print(pipe(prompt_example, max_new_tokens=50)[0]["generated_text"])

Device set to use cuda:0


<|im_start|>system
你是一个非常棒的人工智能助手，是UP主 “用代码打点酱油的chaofa” 开发的。<|im_end|>
<|im_start|>user
天气太热了，所以我今天没有学习一点。
翻译成文言文：<|im_end|>
<|im_start|>assistant
今暑日之气甚盛，则无以适习矣。


In [29]:
from transformers import pipeline

pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer)

prompt_example = """<|im_start|>system
你是一个非常棒的人工智能助手，是UP主 “用代码打点酱油的chaofa” 开发的。<|im_end|>
<|im_start|>user
天气太热了，所以我今天没有学习一点。
翻译成文言文：<|im_end|>
<|im_start|>assistant
"""

print(pipe(prompt_example, max_new_tokens=50)[0]["generated_text"])

Device set to use cuda:0


<|im_start|>system
你是一个非常棒的人工智能助手，是UP主 “用代码打点酱油的chaofa” 开发的。<|im_end|>
<|im_start|>user
天气太热了，所以我今天没有学习一点。
翻译成文言文：<|im_end|>
<|im_start|>assistant
温日之暑，故不能学也。
